# Test the video → 3D box → speed pipeline on BrnoCompSpeed

Runs `scripts/brno_eval_cli.py` on BrnoCompSpeed recordings and scores them with the dataset's official rules
(ported in `speed_lstm/brno.py`): tracks are matched to ground-truth cars by the time they cross the last
measurement line (±0.2 s, same lane), and speed errors are in km/h over valid cars.

Three speeds per matched car:
- **model**: the v2 model on the video-lifted 3D boxes (the real pipeline, trained on I-24 only)
- **geometry_median**: official per-frame median speed of the same lifted trajectory (the official eval.py default)
- **geometry_full**: official line-to-line speed of the same trajectory

Each run also scores the calibration file's own tracks (the Dubska et al. reference system) with the same code
over the same ground-truth cars, as a sanity check and a reference point.

The camera P comes from the recording's `system_dubska_optimal_calib.json` (vanishing points + scale);
the first line of each run checks it against the dataset's hand-measured distances.

Use a GPU runtime (Runtime → Change runtime type → T4).

In [ ]:
# ---- settings ----
RECORDINGS = ['session4_center']   # e.g. add 'session4_left', 'session5_center', ... (sessions 1-6 are test data)
MAX_SECONDS = 600                  # first 10 minutes of each ~1 h video
FRAME_STEP = 2                     # 50 fps -> 25 fps, near the ~30 fps the model was trained on
WEIGHTS = 'yolo11m.pt'
CALIB_SYSTEM = 'dubska_optimal_calib'
BRANCH = 'main'
MODE = '3d'                        # which checkpoint: 2d / 3d / combined
REUSE_DETECTIONS = True            # reuse detections.csv from an earlier run on Drive (same FRAME_STEP)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT = None
for c in ['/content/drive/MyDrive/Final-Project-CHULA', '/content/drive/Shareddrives/Final-Project-CHULA']:
    if os.path.isdir(c):
        PROJECT = c
        break
assert PROJECT, 'Add a shortcut to Final-Project-CHULA in MyDrive, or set PROJECT manually'

def is_brno_root(d):
    return os.path.isdir(f'{d}/dataset') and os.path.isdir(f'{d}/results')

BRNO = None
for c in [f'{PROJECT}/Brno/data/2016-ITS-BrnoCompSpeed', '/content/drive/MyDrive/BrnoCompSpeed/2016-ITS-BrnoCompSpeed',
          '/content/drive/MyDrive/BrnoCompSpeed', '/content/drive/MyDrive/2016-ITS-BrnoCompSpeed']:
    if is_brno_root(c):
        BRNO = c
        break
if BRNO is None:  # search a few levels down
    for root, dirs, _ in os.walk('/content/drive/MyDrive'):
        if root.count(os.sep) > 7:
            dirs[:] = []
        elif is_brno_root(root):
            BRNO = root
            break
assert BRNO, 'Could not find the BrnoCompSpeed folder (with dataset/ and results/); set BRNO manually'
print('PROJECT =', PROJECT)
print('BRNO    =', BRNO)
for rec in RECORDINGS:
    print(rec, os.listdir(f'{BRNO}/dataset/{rec}'))

In [ ]:
%cd /content
!rm -rf capcapcar-speed && git clone -q -b {BRANCH} https://github.com/WalkerCze3/capcapcar-speed.git
%cd capcapcar-speed
!pip install -q -r requirements-video.txt

## Checkpoint

Uses `Final-Project-CHULA/runs/v2/<MODE>/best.pt` from Drive (trained on I-24 by `speed_video_i24_test.ipynb`).
If it's missing, trains one on the I-24 data (~10 min on a T4) and saves it to Drive.

In [ ]:
import shutil
CKPT_DRIVE = f'{PROJECT}/runs/v2/{MODE}/best.pt'
CKPT = f'runs/v2/{MODE}/best.pt'
if os.path.exists(CKPT_DRIVE):
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    shutil.copy(CKPT_DRIVE, CKPT)
    print('using', CKPT_DRIVE)
else:
    I24 = next(root for root, dirs, _ in os.walk(PROJECT) if {'obj', 'ts', 'hg'}.issubset(dirs))
    !python scripts/train_v2_cli.py --data-dir "{I24}" --mode {MODE} --out runs/v2/{MODE} --epochs 10
    os.makedirs(os.path.dirname(CKPT_DRIVE), exist_ok=True)
    shutil.copy(CKPT, CKPT_DRIVE)
    print('saved', CKPT_DRIVE)

## Run: detect + track → 3D boxes → model → official Brno scoring

In [ ]:
import json, subprocess

def sh(cmd):
    # A child process's output doesn't reach the Colab cell on its own; stream it so progress is visible.
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise subprocess.CalledProcessError(p.returncode, cmd)

summaries = {}
for rec in RECORDINGS:
    out = f'runs/brno/{rec}'
    print(f'==================== {rec} ====================')
    cmd = ['python', 'scripts/brno_eval_cli.py',
           '--session-dir', f'{BRNO}/dataset/{rec}',
           '--calib', f'{BRNO}/results/{rec}/system_{CALIB_SYSTEM}.json',
           '--checkpoint', CKPT, '--weights', WEIGHTS,
           '--max-seconds', str(MAX_SECONDS), '--frame-step', str(FRAME_STEP), '--out-dir', out]
    prev = f'{PROJECT}/runs/brno/{rec}/detections.csv'
    if REUSE_DETECTIONS and os.path.exists(prev):  # skip YOLO; re-lift / re-score only
        shutil.copy(prev, '/content/detections_cache.csv')
        cmd += ['--detections', '/content/detections_cache.csv']
        print('reusing', prev)
    sh(cmd)
    summaries[rec] = json.load(open(f'{out}/summary.json'))
    shutil.copytree(out, f'{PROJECT}/runs/brno/{rec}', dirs_exist_ok=True)
print('outputs copied to', f'{PROJECT}/runs/brno')

## Results (km/h)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for rec, s in summaries.items():
    for k in ('model', 'geometry_full', 'geometry_median'):
        rows.append({'recording': rec, 'speed': k, 'matched': f"{s['matched']}/{s['gt_cars_valid']}",
                     **{m: s[k].get(m) for m in ('mean', 'median', 'p95', 'worst', 'bias', 'mean_rel_pct')}})
    ref = s.get('reference_system')
    if ref:
        rows.append({'recording': rec, 'speed': f'reference ({CALIB_SYSTEM}), median mode', 'matched': ref['matched'],
                     **{m: ref['median'].get(m) for m in ('mean', 'median', 'p95', 'worst', 'bias')}})
print(pd.DataFrame(rows).round(2).to_string(index=False))

ev = pd.concat([pd.read_csv(f'runs/brno/{rec}/car_eval.csv').assign(recording=rec) for rec in summaries])
ev = ev[ev['valid'] & ev['matched']]
fig, ax = plt.subplots(figsize=(6, 6))
lim = [min(ev['gt_kmh'].min(), 40) - 5, max(ev[['gt_kmh', 'model_kmh', 'full_kmh']].max()) + 5]
ax.plot(lim, lim, color='gray', lw=1)
ax.scatter(ev['gt_kmh'], ev['model_kmh'], s=12, label='model (video 3D boxes)')
ax.scatter(ev['gt_kmh'], ev['full_kmh'], s=12, alpha=0.6, label='geometry, official line-to-line')
ax.set(xlim=lim, ylim=lim, xlabel='ground-truth speed (km/h)', ylabel='measured (km/h)', title=', '.join(summaries))
ax.legend()
plt.show()